# Editor automático de videos — desde el navegador

No necesitas instalar nada: todo corre en los servidores de Google. Funciona en
cualquier PC por vieja que sea, mientras el navegador abra esta página.

**Cómo se usa:** ejecuta las celdas en orden con el botón ▶ de la izquierda.
La primera tarda unos 2 minutos; las demás son rápidas.

En *Entorno de ejecución → Cambiar tipo de entorno* deja **CPU** (no hace falta GPU).


## 1. Instalar (2 minutos, una vez por sesión)


In [ ]:
#@title Instalar el editor { display-mode: "form" }
RAMA = "claude/auto-video-editing-app-rft92m" #@param {type:"string"}

!apt-get -qq install -y ffmpeg > /dev/null
![ -d /content/lab-editor-videos ] || git clone -q --branch $RAMA https://github.com/jona374/lab-editor-videos /content/lab-editor-videos
%cd /content/lab-editor-videos
!git fetch -q origin $RAMA && git checkout -q $RAMA && git pull -q
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "/content/lab-editor-videos")
print("Listo: ffmpeg y el editor instalados.")


## 2. Tus claves

Se piden ocultas y **no quedan guardadas**: viven solo mientras la pestaña esté abierta.

| Clave | Para qué | Si no la tienes |
|---|---|---|
| `ELEVENLABS_API_KEY` | la voz en off | marca **SIN_VOZ** al generar |
| `ANTHROPIC_API_KEY` | que Claude escriba el guion | pega tú el guion |


In [ ]:
#@title Pegar claves (Enter para saltar la que no tengas) { display-mode: "form" }
import os
from getpass import getpass

for nombre in ("ELEVENLABS_API_KEY", "ANTHROPIC_API_KEY"):
    valor = getpass(f"{nombre}: ").strip()
    if valor:
        os.environ[nombre] = valor

print("Voz en off:", "sí" if os.getenv("ELEVENLABS_API_KEY") else "no")
print("Guion automático:", "sí" if os.getenv("ANTHROPIC_API_KEY") else "no (pégalo tú)")


## 3. Elegir la voz

Lista las voces de tu cuenta de ElevenLabs. Copia el ID de la que quieras
y pégalo en la celda 6. Para español, busca voces multilingües.


In [ ]:
#@title Ver mis voces de ElevenLabs { display-mode: "form" }
from editor import voz as voces

try:
    lista = voces.voces()
    if not lista:
        print("No hay voces (¿falta la clave?)")
    for v in lista:
        print(f"{v['id']}   {v['nombre']}   {v['idioma']}")
except Exception as error:
    print("No pude listar las voces:", error)


## 4. Traer tus clips desde Google Drive

Si tus videos ya están en Drive, no hace falta volver a subirlos.
Al ejecutar la celda, Google te pide permiso para conectar tu Drive: acepta.

Escribe en `CARPETA` el **nombre** de la carpeta de Drive donde están los clips
(por ejemplo `clips de prueba textiles pelileo`).


In [ ]:
#@title Copiar clips desde Drive { display-mode: "form" }
CARPETA = "clips de prueba textiles pelileo" #@param {type:"string"}
MARCA = "textiles-pelileo" #@param {type:"string"}
TOMAS_DE_APOYO = False #@param {type:"boolean"}

import shutil
from pathlib import Path
from google.colab import drive
from editor import marcas
from editor.utilidades import EXTENSIONES_VIDEO, duracion

drive.mount("/content/drive", force_remount=False)

# Busca la carpeta por nombre (ignora mayúsculas y espacios sobrantes).
# Primero donde suele estar, y si no, hasta 3 niveles hacia abajo: recorrer todo
# el Drive sería lentísimo.
buscada = CARPETA.strip().lower()
raiz = Path("/content/drive/MyDrive")
origen = next((c for c in raiz.iterdir() if c.is_dir() and c.name.strip().lower() == buscada), None)
if origen is None:
    for patron in ("*/*", "*/*/*"):
        origen = next((c for c in raiz.glob(patron)
                       if c.is_dir() and c.name.strip().lower() == buscada), None)
        if origen:
            break

if origen is None:
    raise SystemExit(f"No encontré una carpeta llamada {CARPETA!r} en tu Drive. "
                     "Revisa el nombre o usa la celda de subir archivos a mano.")

marca = marcas.obtener(MARCA, crear=True)
destino = marca.tomas_de_apoyo if TOMAS_DE_APOYO else marca.por_editar
print(f"Copiando desde {origen} …  (los archivos grandes tardan)")

for archivo in sorted(origen.iterdir()):
    if archivo.is_file() and archivo.suffix.lower() in EXTENSIONES_VIDEO:
        copia = destino / archivo.name
        if not copia.exists():
            shutil.copy2(archivo, copia)
        print(f"  {archivo.name}  —  {duracion(copia):.0f}s")

print(f"\nListo: {len(marca.clips())} clip(s) por editar, {len(marca.clips_de_apoyo())} de apoyo")


### Alternativa: subir clips desde la computadora

Solo si no los tienes en Drive. Se abre un botón *Elegir archivos*.


In [ ]:
#@title Subir clips a mano { display-mode: "form" }
MARCA = "textiles-pelileo" #@param {type:"string"}
import shutil
from pathlib import Path
from google.colab import files
from editor import marcas

marca = marcas.obtener(MARCA, crear=True)
for nombre in files.upload():
    shutil.move(nombre, marca.por_editar / Path(nombre).name)

print(f"{len(marca.clips())} clip(s) por editar")


## 5. Música (opcional)

Sube uno o varios mp3. Si no subes nada, el video sale sin música.


In [ ]:
#@title Subir música { display-mode: "form" }
AMBIENTE = "energico" #@param ["energico", "inspirador", "calmado", "urbano", "elegante"]

import shutil
from pathlib import Path
from google.colab import files

carpeta = Path("/content/lab-editor-videos/assets/musica") / AMBIENTE
carpeta.mkdir(parents=True, exist_ok=True)
for nombre in files.upload():
    shutil.move(nombre, carpeta / Path(nombre).name)

!ls -R /content/lab-editor-videos/assets/musica


## 6. Hacer el video

Llena **TEMA** (si tienes clave de Claude) **o** pega tu guion en **GUION**.
Si llenas los dos, manda el guion.

Un video de 46 s tarda entre 1 y 3 minutos en salir.


In [ ]:
#@title Generar { display-mode: "form" }
TEMA = "" #@param {type:"string"}
GUION = "" #@param {type:"string"}
DURACION = 46 #@param [30, 46, 60, 90] {type:"raw"}
CORTE_CADA = 2 #@param [1.5, 2, 3] {type:"raw"}
FORMATO = "vertical" #@param ["vertical", "cuadrado", "horizontal"]
VOZ_ID = "" #@param {type:"string"}
SIN_VOZ = False #@param {type:"boolean"}
SUBTITULOS = True #@param {type:"boolean"}

from editor.config import Ajustes
from editor.pipeline import crear_video

ajustes = Ajustes(
    marca=MARCA,
    duracion_objetivo=float(DURACION),
    duracion_corte=float(CORTE_CADA),
    formato=FORMATO,
    voz_id=VOZ_ID,
    subtitulos=SUBTITULOS,
)
resultado = crear_video(
    ajustes,
    tema=TEMA or None,
    texto_guion=GUION or None,
    sin_voz=SIN_VOZ,
    avisar=print,
)
print("\nVideo:", resultado.video)


## 7. Ver y descargar


In [ ]:
#@title Ver el video aquí mismo { display-mode: "form" }
import base64
from IPython.display import HTML

datos = base64.b64encode(resultado.video.read_bytes()).decode()
HTML(f'<video width="320" controls src="data:video/mp4;base64,{datos}"></video>')


In [ ]:
#@title Descargar el video, el guion y los subtítulos { display-mode: "form" }
from google.colab import files

files.download(str(resultado.video))
files.download(str(resultado.carpeta / "guion.md"))
if resultado.subtitulos:
    files.download(str(resultado.subtitulos))


---

### Si algo falla

| Dice | Qué pasa |
|---|---|
| `no hay clips en …` | falta ejecutar la celda 4 (traer clips) |
| `No encontré una carpeta llamada …` | revisa el nombre exacto de la carpeta en Drive |
| `falta ELEVENLABS_API_KEY` | vuelve a la celda 2, o marca **SIN_VOZ** |
| `falta la voz` | pega un **VOZ_ID** de los que salieron en la celda 3 |
| `falta ANTHROPIC_API_KEY` | pega el guion en **GUION** en vez de usar **TEMA** |
| se reinició y perdió todo | Colab se apaga solo tras un rato inactivo: corre otra vez desde la celda 1 |

Para hacer otro video con los mismos clips: cambia el guion en la celda 6 y ejecútala de nuevo.
Con `--semilla` (en `Ajustes`) puedes repetir exactamente el mismo montaje y cambiar solo la voz.
